# MFScope — Model Exploration

This notebook walks through:
1. Loading feature data from the DB
2. EDA — distribution of features, correlation matrix, category breakdowns
3. Rule-based scorer audit — which features drive scores
4. XGBoost v2 training — forward label construction, time-split, evaluation
5. SHAP interpretation — global + per-fund feature importance
6. Backtesting sanity check — do high-scored funds outperform?

> **Prerequisite**: Run the ingestion pipeline first so the DB has data.
> ```bash
> uvicorn backend.api.main:app --reload &
> curl -X POST http://localhost:8000/api/v1/admin/refresh
> ```

In [ ]:
import sys
sys.path.insert(0, '..')  # allow importing from project root

import asyncio
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Dark-ish plot style to match the dashboard
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0A0B0D',
    'axes.facecolor':   '#111318',
    'axes.edgecolor':   '#1E2028',
    'axes.labelcolor':  '#8B8FA8',
    'xtick.color':      '#555870',
    'ytick.color':      '#555870',
    'grid.color':       '#1E2028',
    'text.color':       '#E8EAF0',
    'font.family':      'sans-serif',
})

AMBER  = '#F59E0B'
GREEN  = '#10B981'
RED    = '#EF4444'
GRAY   = '#6B7280'

print('Imports OK')

## 1 — Load feature data from DB

In [ ]:
from sqlalchemy import select, text
from backend.db.session import AsyncSessionLocal
from backend.db.models import FundFeatures, FundScore, Scheme

async def load_features() -> pd.DataFrame:
    async with AsyncSessionLocal() as session:
        result = await session.execute(
            select(FundFeatures, Scheme.scheme_name, Scheme.category, Scheme.amc_name)
            .join(Scheme, FundFeatures.scheme_id == Scheme.id)
            .order_by(FundFeatures.feature_date.desc())
        )
        rows = result.all()

    records = []
    for feat, name, cat, amc in rows:
        d = {c.name: getattr(feat, c.name) for c in feat.__table__.columns}
        d['scheme_name'] = name
        d['category']    = cat
        d['amc_name']    = amc
        records.append(d)

    return pd.DataFrame(records)

df = asyncio.run(load_features())
print(f'Loaded {len(df):,} feature rows across {df["scheme_id"].nunique():,} schemes')
df.head(3)

In [ ]:
async def load_scores() -> pd.DataFrame:
    async with AsyncSessionLocal() as session:
        result = await session.execute(
            select(FundScore, Scheme.scheme_name, Scheme.category)
            .join(Scheme, FundScore.scheme_id == Scheme.id)
            .order_by(FundScore.score_date.desc())
        )
        rows = result.all()

    records = []
    for score, name, cat in rows:
        d = {c.name: getattr(score, c.name) for c in score.__table__.columns}
        d['scheme_name'] = name
        d['category']    = cat
        records.append(d)
    return pd.DataFrame(records)

scores_df = asyncio.run(load_scores())
print(f'Loaded {len(scores_df):,} score rows')
scores_df.head(3)

## 2 — EDA

In [ ]:
# ── Feature completeness ──────────────────────────────────────────────────────
FEATURE_COLS = [
    'return_1m', 'return_3m', 'return_6m', 'return_1y', 'return_3y',
    'volatility_1y', 'sharpe_1y', 'sortino_1y', 'alpha_1y', 'beta_1y',
    'max_drawdown_1y', 'momentum_roc_1m', 'momentum_roc_3m', 'ma_crossover',
    'expense_ratio', 'aum_crore', 'aum_growth_3m',
    'manager_tenure_years', 'category_rank_pct',
    'sentiment_7d', 'sentiment_30d', 'news_volume_7d',
]

# Latest snapshot per scheme
latest = df.sort_values('feature_date').groupby('scheme_id').last().reset_index()

completeness = (
    latest[FEATURE_COLS].notna().mean().sort_values(ascending=True) * 100
)

fig, ax = plt.subplots(figsize=(8, 7))
completeness.plot.barh(ax=ax, color=AMBER, alpha=0.85)
ax.set_xlabel('% of schemes with value', labelpad=8)
ax.set_title('Feature completeness (latest snapshot)', pad=12, fontsize=12)
ax.axvline(80, color=GRAY, linestyle='--', linewidth=0.8, label='80% threshold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Score distribution by category ───────────────────────────────────────────
if len(scores_df) > 0:
    latest_scores = scores_df.sort_values('score_date').groupby('scheme_id').last().reset_index()

    top_cats = latest_scores['category'].value_counts().head(8).index
    plot_df  = latest_scores[latest_scores['category'].isin(top_cats)]

    fig, ax = plt.subplots(figsize=(10, 4))
    for cat in top_cats:
        subset = plot_df[plot_df['category'] == cat]['composite_score']
        ax.hist(subset, bins=20, alpha=0.5, label=cat, density=True)

    ax.set_xlabel('Composite score (0–100)')
    ax.set_ylabel('Density')
    ax.set_title('Score distribution by category', pad=10, fontsize=12)
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()
else:
    print('No score data yet — run the scoring pipeline first.')

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
corr_cols = [
    'return_1y', 'sharpe_1y', 'sortino_1y', 'volatility_1y',
    'alpha_1y', 'beta_1y', 'max_drawdown_1y',
    'expense_ratio', 'aum_crore', 'sentiment_7d',
]

corr = latest[corr_cols].dropna(how='all').corr()

fig, ax = plt.subplots(figsize=(8, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(10, 145, s=80, l=40, as_cmap=True)
sns.heatmap(
    corr, mask=mask, cmap=cmap, vmin=-1, vmax=1,
    linewidths=0.4, linecolor='#1E2028',
    annot=True, fmt='.2f', annot_kws={'size': 8},
    ax=ax,
)
ax.set_title('Feature correlation matrix (latest snapshot)', pad=12, fontsize=12)
plt.tight_layout()
plt.show()

## 3 — Rule-based scorer audit

In [ ]:
# Parse component scores from scores_df
if len(scores_df) > 0:
    comp_cols = ['score_returns', 'score_consistency', 'score_cost', 'score_sentiment', 'score_stability']
    weights   = [0.40, 0.20, 0.15, 0.15, 0.10]

    latest_scores = scores_df.sort_values('score_date').groupby('scheme_id').last().reset_index()

    means = latest_scores[comp_cols].mean()
    weighted_contribution = means * weights

    labels = ['Returns\n(40%)', 'Consistency\n(20%)', 'Cost\n(15%)', 'Sentiment\n(15%)', 'Stability\n(10%)']
    colors = [AMBER, GREEN, '#60A5FA', '#A78BFA', GRAY]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].bar(labels, means, color=colors, alpha=0.85)
    axes[0].set_ylim(0, 100)
    axes[0].set_title('Average component score', fontsize=11)
    axes[0].set_ylabel('Score (0–100)')

    axes[1].bar(labels, weighted_contribution, color=colors, alpha=0.85)
    axes[1].set_title('Weighted contribution to composite', fontsize=11)
    axes[1].set_ylabel('Points contributed')

    plt.suptitle('Rule-based scorer — component breakdown', y=1.02, fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('No score data yet.')

## 4 — XGBoost v2: forward label construction + training

In [ ]:
from datetime import date, timedelta
from backend.scoring.ml_model import MLScorer, FEATURE_COLS

scorer = MLScorer()

# Use data up to 6 months ago as cutoff so forward labels can be computed
cutoff = date.today() - timedelta(days=180 + 30)
print(f'Training cutoff: {cutoff}')

# Build dataset
X, y = asyncio.run(scorer.build_training_data(cutoff_date=cutoff, forward_days=180))
print(f'Dataset shape: X={X.shape}, y={y.shape}')
print(f'Forward Sharpe  — mean: {y.mean():.3f}  std: {y.std():.3f}  range: [{y.min():.2f}, {y.max():.2f}]')

y.hist(bins=40, color=AMBER, alpha=0.8, edgecolor='#0A0B0D', figsize=(7,3))
plt.title('Distribution of forward 6-month Sharpe (training labels)', fontsize=11)
plt.xlabel('Sharpe ratio')
plt.tight_layout()
plt.show()

In [ ]:
# ── Train the model ───────────────────────────────────────────────────────────
metrics = asyncio.run(scorer.train(cutoff_date=cutoff))
print('Training metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

## 5 — SHAP feature importance

In [ ]:
import shap

scorer.load()
X_sample = X.fillna(X.median()).sample(min(500, len(X)), random_state=42)

explainer   = shap.TreeExplainer(scorer._model)
shap_values = explainer.shap_values(X_sample)

shap.summary_plot(
    shap_values, X_sample,
    feature_names=FEATURE_COLS,
    plot_type='bar',
    max_display=15,
    show=False,
)
plt.title('XGBoost — global SHAP feature importance', fontsize=12, pad=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Beeswarm (full impact + direction) ───────────────────────────────────────
shap.summary_plot(
    shap_values, X_sample,
    feature_names=FEATURE_COLS,
    max_display=15,
    show=False,
)
plt.title('SHAP beeswarm — feature impact direction', fontsize=12, pad=10)
plt.tight_layout()
plt.show()

## 6 — Backtesting sanity check
Do funds scored in the top quartile (Strong Buy) actually outperform bottom quartile (Strong Sell) funds over the following 6 months?

In [ ]:
# Merge features with scores, then join forward return from NAV
# (simplified: use return_1y as a proxy since full backtest needs point-in-time NAV lookups)

if len(scores_df) > 0 and len(df) > 0:
    merged = latest_scores.merge(
        latest[['scheme_id', 'return_1y', 'return_3y', 'sharpe_1y', 'category']],
        on='scheme_id', how='inner', suffixes=('', '_feat')
    )

    merged['quartile'] = pd.qcut(
        merged['composite_score'], q=4,
        labels=['Q1 (Sell)', 'Q2', 'Q3', 'Q4 (Buy)']
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    for ax, col, label in [
        (axes[0], 'return_1y', '1Y Return (%)'),
        (axes[1], 'sharpe_1y', '1Y Sharpe Ratio'),
    ]:
        data = [merged[merged['quartile'] == q][col].dropna() for q in merged['quartile'].cat.categories]
        bp = ax.boxplot(data, patch_artist=True, notch=True, showfliers=False)
        for patch, color in zip(bp['boxes'], [RED, GRAY, GREEN, AMBER]):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_xticklabels(merged['quartile'].cat.categories, fontsize=8)
        ax.set_ylabel(label)
        ax.set_title(f'{label} by score quartile', fontsize=10)
        ax.axhline(0, color=GRAY, linewidth=0.6, linestyle='--')
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle('Score quartile vs realised performance (sanity check)', y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()

    # Summary table
    summary = merged.groupby('quartile', observed=True)[['return_1y', 'sharpe_1y']].agg(['mean', 'median']).round(2)
    print(summary)
else:
    print('Need both features and scores in the DB to run this check.')

## 7 — Per-fund SHAP waterfall (single prediction explainer)

In [ ]:
# Pick the top-scored fund and explain its prediction
if len(scores_df) > 0 and scorer._model is not None:
    top_row = merged.nlargest(1, 'composite_score').iloc[0]
    print(f"Explaining: {top_row.get('scheme_name', 'N/A')} | Score: {top_row['composite_score']:.1f} | {top_row['conviction']}")

    feat_row = latest[latest['scheme_id'] == top_row['scheme_id']][FEATURE_COLS]
    feat_row = feat_row.fillna(feat_row.median()).iloc[[0]]

    explainer_single = shap.TreeExplainer(scorer._model)
    sv = explainer_single(feat_row)

    shap.plots.waterfall(sv[0], max_display=12, show=False)
    plt.title(f'SHAP waterfall — {top_row.get("scheme_name", "")[:50]}', fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print('Load the trained model and ensure score data exists to run this cell.')